# Checkpoint A1: Cài đặt LM Studio + Hermes + thiết lập môi trường

Chạy từng cell theo thứ tự để cài đặt và thiết lập toàn bộ môi trường.
Sau khi xong, tiếp tục từ **Bước A2: Khởi tạo 3 Agent Profiles**.

## A1.1 — Hướng dẫn cài đặt LM Studio

LM Studio là một ứng dụng giao diện đồ họa trực quan giúp bạn tải và chạy các mô hình ngôn ngữ lớn (LLM) cục bộ một cách dễ dàng.

### Hướng dẫn tải và cài đặt LM Studio:
1. Truy cập trang chủ [LM Studio](https://lmstudio.ai/) và tải bản cài đặt phù hợp với hệ điều hành của bạn (Windows, macOS, hoặc Linux).
2. Chạy file cài đặt đã tải xuống và làm theo hướng dẫn trên màn hình để hoàn tất.
3. Sau khi cài đặt thành công, hãy mở ứng dụng LM Studio lên.

In [ ]:
# Kiểm tra kết nối tới LM Studio Local Server (mặc định cổng 1234)
# Hướng dẫn bật Local Server trong LM Studio:
# 1. Mở LM Studio.
# 2. Chọn tab "Local Server" (biểu tượng <-> ở thanh công cụ bên trái).
# 3. Chọn model đã tải ở menu dropdown trên cùng.
# 4. Nhấn nút "Start Server".

import urllib.request
import json

try:
    req = urllib.request.Request('http://localhost:1234/v1/models')
    with urllib.request.urlopen(req, timeout=3) as resp:
        data = json.loads(resp.read().decode())
    print("LM Studio Local Server đang chạy thành công tại http://localhost:1234/v1!")
except Exception as e:
    print("LM Studio Local Server chưa chạy hoặc chưa được bật ở cổng 1234.")
    print("Hãy mở LM Studio, bật Local Server và chạy lại cell này.")

## A1.2 — Tải mô hình gemma4-e2b và kiểm tra trạng thái load

Trong LM Studio, hãy thực hiện tìm kiếm và tải mô hình **gemma4-e2b** (mô hình local siêu nhẹ thế hệ mới từ Google tối ưu tốt cho tiếng Việt và lý luận). Sau khi tải xong, chọn tải mô hình này lên Local Server trong LM Studio.

In [ ]:
# Kiểm tra mô hình đã được tải và sẵn sàng inference chưa
import urllib.request
import json

try:
    # 1. Kiểm tra danh sách mô hình đang được load
    req = urllib.request.Request('http://localhost:1234/v1/models')
    with urllib.request.urlopen(req, timeout=3) as resp:
        models_data = json.loads(resp.read().decode())
    
    loaded_models = [m['id'] for m in models_data.get('data', [])]
    if not loaded_models:
        print("Cảnh báo: Server đang chạy nhưng chưa có mô hình nào được load.")
        print("Vui lòng chọn mô hình gemma4:e2b ở menu dropdown trên cùng trong LM Studio Local Server.")
    else:
        print(f"Các mô hình đang được load trong LM Studio: {loaded_models}")
        
        # 2. Gửi request test chat completion (inference)
        test_payload = {
            "model": loaded_models[0],
            "messages": [
                {"role": "user", "content": "Xin chào, hãy giới thiệu ngắn gọn về bản thân bạn."}
            ],
            "temperature": 0.7,
            "max_tokens": 100
        }
        
        headers = {'Content-Type': 'application/json'}
        req_chat = urllib.request.Request(
            'http://localhost:1234/v1/chat/completions',
            data=json.dumps(test_payload).encode('utf-8'),
            headers=headers,
            method='POST'
        )
        
        print("\nĐang gửi câu hỏi kiểm tra tới mô hình...")
        with urllib.request.urlopen(req_chat, timeout=10) as resp_chat:
            chat_data = json.loads(resp_chat.read().decode())
            
        response_text = chat_data['choices'][0]['message']['content']
        print("\n[SUCCESS] Mô hình đã phản hồi thành công!")
        print("="*40)
        print(response_text)
        print("="*40)
        print("Mô hình gemma4:e2b đã sẵn sàng phục vụ!")
except Exception as e:
    print(f"Lỗi khi kiểm tra mô hình: {e}")
    print("Vui lòng đảm bảo LM Studio Local Server đang chạy và mô hình gemma4:e2b đã được chọn.")

## A1.3 — Cài đặt Hermes Agent

Hermes là một framework tác tử AI gọn nhẹ, hỗ trợ gọi mô hình cục bộ qua endpoint tương thích OpenAI.

### Hướng dẫn cài đặt:

**Windows (Khuyến nghị sử dụng Virtual Environment):**
1. Kích hoạt môi trường ảo `.venv` trong PowerShell:
   ```powershell
   .\.venv\bin\Activate.ps1
   ```
2. Thực hiện cài đặt `hermes-agent` bằng `pip`:
   ```powershell
   pip install hermes-agent
   ```
3. Kiểm tra xem lệnh `hermes` có chạy được không:
   ```powershell
   hermes --version
   ```

**macOS (Khuyến nghị dùng Homebrew tránh lỗi PEP 668):**
```bash
brew install hermes-agent
```

**Linux / WSL2:**
```bash
pip install --user hermes-agent
# Hoặc sử dụng pipx (an toàn và độc lập hơn):
pipx install hermes-agent
```

In [ ]:
# Kiểm tra Hermes Agent
import shutil
import os
import subprocess

# Thử tìm hermes trong PATH hoặc trong thư mục ảo .venv
hermes_path = shutil.which('hermes')
if not hermes_path:
    for relative_path in ['.venv/bin/hermes', '.venv/Scripts/hermes.exe', '.venv/bin/hermes.exe']:
        if os.path.exists(relative_path):
            hermes_path = relative_path
            break

if hermes_path:
    try:
        result = subprocess.run([hermes_path, '--version'], capture_output=True, text=True)
        print(f'Hermes Agent đã cài đặt thành công: {result.stdout.strip()}')
    except Exception as e:
        print(f'Lỗi khi chạy lệnh hermes: {e}')
else:
    print('Chưa tìm thấy lệnh hermes.')
    print('Vui lòng kích hoạt virtual environment và chạy lệnh: pip install hermes-agent')

## A1.4 — Cấu hình Hermes kết nối với LM Studio

In [ ]:
# Khởi tạo Hermes kết nối với LM Studio Local Server
import os
import json
import urllib.request

config_dir = os.path.expanduser('~/.hermes')
config_path = os.path.join(config_dir, 'config.yaml')

os.makedirs(config_dir, exist_ok=True)

# Xác định model từ danh sách LM Studio đang load
model_name = 'gemma4-e2b'
try:
    req = urllib.request.Request('http://localhost:1234/v1/models')
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read().decode())
    loaded_models = [m['id'] for m in data.get('data', [])]
    if loaded_models:
        model_name = loaded_models[0]
except Exception:
    pass

config_content = f"""model:
  default: {model_name}
  provider: custom
  base_url: http://localhost:1234/v1
providers: {{}}
"""

with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_content)

print(f'Đã ghi config: {config_path}')
print(f'Model: {model_name}')
print(f'Base URL: http://localhost:1234/v1')

In [ ]:
# Test Hermes chat nhanh
# Chú ý: cell này cần hermes CLI, có thể cần chạy thủ công
# hermes chat → gõ: "Bạn đang dùng mô hình nào?" → /exit

import subprocess
try:
    result = subprocess.run(['hermes', 'chat', '--help'], 
                          capture_output=True, text=True, timeout=5)
    print('Hermes chat sẵn sàng.')
    print('Chạy thử trong terminal dòng lệnh: hermes chat')
except FileNotFoundError:
    print('Hermes chưa cài. Quay lại bước A1.3.')
except Exception as e:
    print(f'Lỗi: {e}')

## A1.5 — Khởi tạo cấu trúc thư mục + tài liệu mô phỏng

In [ ]:
# Tạo cấu trúc thư mục thực hành
import os

base_dir = os.path.expanduser('../../output/vtn-session05')
subdirs = ['templates', 'runs', 'synthetic-data',
           'templates/skills', 'templates/skills/vtn-agent-skill',
           'templates/skills/vtn-agent-skill/kb',
           'templates/skills/vtn-agent-skill/scripts',
           'templates/skills/vtn-agent-skill/schemas']

for subdir in subdirs:
    full_path = os.path.join(base_dir, subdir)
    os.makedirs(full_path, exist_ok=True)

print(f'Đã tạo cấu trúc thư mục tại: {base_dir}')
for s in subdirs:
    print(f'  {s}/')

In [ ]:
# Tạo thư mục mô phỏng /docs/simulated và /drafts
import os

# Tạo tài liệu BGP mô phỏng
docs_dir = os.path.expanduser('../../output/vtn-session05/simulated-docs')
os.makedirs(docs_dir, exist_ok=True)

bgp_content = '''# Tài liệu mô phỏng: cấu hình BGP cơ bản tại VTN

## 1. Mục đích
Tài liệu này mô tả khái niệm cơ bản về BGP và quy trình cấu hình mô phỏng
dành cho bài lab đào tạo. Không dùng cho hệ thống thật.

## 2. BGP là gì
BGP (Border Gateway Protocol) là giao thức định tuyến dùng để trao đổi
thông tin định tuyến giữa các hệ tự trị (Autonomous System) trên mạng diện rộng.

## 3. Quy trình cấu hình BGP mô phỏng
1. Kiểm tra trạng thái router trước thay đổi.
2. Xác định số AS nội bộ và AS láng giềng.
3. Khai báo tiến trình BGP mô phỏng.
4. Khai báo neighbor mô phỏng.
5. Kiểm tra trạng thái phiên BGP.
6. Ghi log kết quả kiểm tra.

## 4. Điều kiện dừng
Nếu phiên BGP không lên trạng thái Established trong thời gian kiểm thử,
dừng thao tác và chuyển cho kỹ sư vận hành bậc 2.

## 5. Lưu ý an toàn
Không áp dụng trực tiếp nội dung này lên thiết bị thật.
'''

bgp_path = os.path.join(docs_dir, 'vtn_bgp_config_sim.md')
with open(bgp_path, 'w', encoding='utf-8') as f:
    f.write(bgp_content)

print(f'Đã tạo tài liệu BGP: {bgp_path}')
print(f'Kích thước: {len(bgp_content)} bytes')

## A1.6 — Kiểm tra tổng kết

In [ ]:
# Verify toàn bộ môi trường
import os, json, urllib.request, shutil

checks = []

# 1. LM Studio local server
try:
    req = urllib.request.Request('http://localhost:1234/v1/models')
    with urllib.request.urlopen(req, timeout=3) as resp:
        data = json.loads(resp.read().decode())
    checks.append(('LM Studio Local Server', 'data' in data))
except Exception:
    checks.append(('LM Studio Local Server', False))

# 2. Model LM Studio đã load
try:
    req = urllib.request.Request('http://localhost:1234/v1/models')
    with urllib.request.urlopen(req, timeout=3) as resp:
        data = json.loads(resp.read().decode())
    models = [m['id'] for m in data.get('data', [])]
    checks.append(('Model LM Studio', len(models) > 0))
except Exception:
    checks.append(('Model LM Studio', False))

# 3. Hermes CLI
try:
    hermes_path = shutil.which('hermes')
    if not hermes_path:
        for relative_path in ['.venv/bin/hermes', '.venv/Scripts/hermes.exe', '.venv/bin/hermes.exe']:
            if os.path.exists(relative_path):
                hermes_path = relative_path
                break
    checks.append(('Hermes CLI', hermes_path is not None))
except Exception:
    checks.append(('Hermes CLI', False))

# 4. Hermes config
config_path = os.path.expanduser('~/.hermes/config.yaml')
checks.append(('Hermes config', os.path.exists(config_path)))

# 5. Thư mục thực hành
base_dir = os.path.expanduser('../../output/vtn-session05')
checks.append(('Thư mục ../../output/vtn-session05', os.path.isdir(base_dir)))

# 6. Tài liệu BGP
bgp_path = os.path.join(base_dir, 'simulated-docs/vtn_bgp_config_sim.md')
checks.append(('Tài liệu BGP mô phỏng', os.path.exists(bgp_path)))

print('=== KIỂM TRA MÔI TRƯỜNG ===')
all_pass = True
for name, ok in checks:
    icon = 'PASS' if ok else 'FAIL'
    print(f'  [{icon}] {name}')
    if not ok:
        all_pass = False

if all_pass:
    print(f'\nMôi trường sẵn sàng. Tiếp tục Bước A2.')
else:
    print(f'\nCòn mục FAIL. Chạy lại các cell tương ứng ở trên.')

---

**Tiếp tục:** Quay lại `lab.md` → **Bước A2: Khởi tạo 3 Agent Profiles + thiết kế SOUL.md**